# Genuine Cross-Deployment Zero-Label SAE-Handle Transfer + Break-Even K\* + KL Localization

**Artifact `art_hDsehctoIpVZ` — experiment (iter-10, "M2′′′′′")**

The parent research treats sparse-autoencoder (SAE) latents as a learned knowledge
representation and asks whether a **single label-free SAE "absorber" latent** can act as a
reliable *where-to-gate* handle on a deployment it was never tuned on, beating a dense probe
that *does* get to see that deployment's labels.

This experiment is the **evidence-fix** for that claim. It removes a circularity in the
previous iteration (the absorber id was discovered once with full labels, and *both* the
label-free SAE gate and the n-label dense gate were then scored on the **same** eval fold).
It demonstrates **genuine cross-deployment transfer**: the absorber id is *fixed* on
deployment **A**, then on a **disjoint deployment B** the n-independent fixed-id SAE firing
gate (0 deploy labels) is scored against a **fresh dense gate fit on B's own n labels**.

Four pieces (per case):
- **(A/B)** transfer curves on B + break-even **K\* = D / n\*** (one-time discovery labels ÷ dense labels needed to match the SAE handle).
- **(C)** a *selection-independent* next-token **behavioral-KL** localization test vs a random-latent null.
- **(D)** the Amazon `adv_joint`-vs-`adv_pres` **instrument-disagreement** diagnosis at matched forget.

### What this notebook does
The heavy half of `transfer.py` (encoding `gemma-2-2b` + Gemma Scope 16k SAE activations on a
GPU, plus LLM-judged edits) is **pre-computed** and shipped in `mini_demo_data.json`. This demo
loads those results and re-runs the experiment's **pure-numpy decision / aggregation layer
verbatim** — the CI-separation → `n_breakeven` → `K*` → transfer verdict logic, the KL
localization rule, the caveat materiality rule, and the fork `aggregate()` — reproducing every
headline verdict, then visualizes the transfer curves. No GPU required; runs in seconds.

In [ ]:
# --- Install dependencies (Colab-safe) ---
# Every package the notebook uses (numpy, matplotlib, pandas) is pre-installed on Colab, so on
# Colab we install NOTHING (installing would corrupt Colab's pre-loaded C extensions). Locally
# we install them at Colab's exact versions so the environment matches.
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0')

In [ ]:
# --- Imports (original transfer.py used: os, sys, json, numpy; + matplotlib/pandas for the demo) ---
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# numpy 2.0 compat shims (harmless; present in case an older helper uses removed aliases)
if not hasattr(np, "alltrue"):  np.alltrue = np.all
if not hasattr(np, "product"):  np.product = np.prod

print("numpy", np.__version__)

In [ ]:
# --- Data loading helper: GitHub URL (Colab) with local fallback (offline / local Jupyter) ---
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-7ee30c-catching-silent-feature-absorption-in-fr/main/round-10/experiment-3/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
# --- Load the pre-computed experiment output ---
data = load_data()
md_block = data["metadata"]
per_case = md_block["per_case"]                     # the analysis source: one entry per case
datasets = {d["dataset"]: d["examples"] for d in data["datasets"]}

print("model :", md_block["model"], "| SAE:", md_block["sae"]["sae_params"])
print("cases :", list(per_case.keys()))
print("stored fork verdict :", md_block["overall_transfer_fork_verdict"])
print("stored spend        : $%.4f" % md_block["cost"]["usd_spent"])
print("datasets            :", {k: len(v) for k, v in datasets.items()})

In [ ]:
# ============================ CONFIG (tunable) ============================
# These are the decision thresholds copied VERBATIM from transfer.py. Changing them changes the
# verdicts the analysis layer assigns to the (pre-computed) transfer curves.
N_GRID_T       = [1, 5, 20, "full"]   # dense-gate label budgets per deployment (curve x-axis)
K_STAR_SMALL   = 3.0                  # break-even K* below which break-even alone counts as a saving
MATERIAL_THR   = 0.30                 # |forget_gap| below this => Amazon edit caveat is IMMATERIAL
SAE_STRONG_THR = 0.70                 # fixed-id SAE handle balanced-accuracy floor for a transfer claim
KL_NULL_PCTL   = 90                   # robust empirical-null bar = this percentile of the random-latent null
KL_FRAC_THR    = 0.10                 # one-sided permutation p-value ceiling for "localized"

# START MINIMAL: analyze just 1 case to verify the pipeline; scale up to all 5 below.
MAX_CASES = 5                         # 1 (minimal) ... 5 (all cases). Pure analysis -> all 5 run in <1s.

cases = list(per_case.keys())[:MAX_CASES]
print("analyzing %d case(s): %s" % (len(cases), cases))

## Piece A/B — Cross-deployment transfer verdict + break-even K\*

For each case we reconstruct the two gating curves stored in `per_case[cid]["transferB"]`:

- the **fixed-id SAE handle** (one balanced-accuracy point + CI; **0 deployment labels**), and
- the **dense fair gate** fit fresh on B's own `n ∈ {1, 5, 20, full}` labels (a balacc + CI per `n`).

The function below is the **verbatim decision block from `transfer.py:transfer_arm`** — it
computes, from those CIs:
- `dense_below_sae[n]` = dense CI strictly below the SAE handle (the SAE wins at that label budget),
- `n_breakeven` = smallest `n` whose dense CI **overlaps** the SAE handle (dense "catches up"),
- `K* = D_full / n*`, and the **verdict**:
  `TRANSFER_CONFIRMED` (low-n CI separation) / `TRANSFER_VIA_BREAKEVEN` (small K\*) /
  `NO_TRANSFER` / `SAE_WEAK_DESCRIPTIVE` / `UNDERPOWERED`.

In [ ]:
def reproduce_transfer_verdict(summary):
    """Verbatim ci_separation / n_breakeven / K_star / verdict block from transfer.py:transfer_arm,
    fed the pre-computed SAE value+CI and dense per-n value+CI from metadata.per_case[cid]."""
    sae_val = summary["sae_balacc"]
    sae_lo, sae_hi = summary["sae_ci"]
    powered = summary["powered"]
    D_full = summary["D_full"]
    dense_curve = summary["dense"]                  # keys are strings: "1","5","20","full"

    # n_breakeven = smallest n whose dense CI OVERLAPS the SAE flat value (dense "catches up")
    sep, n_breakeven = {}, None
    for n in N_GRID_T:
        key = str(n)
        if key not in dense_curve:
            continue
        lo, hi = dense_curve[key]["ci"]
        overlap = not (hi < sae_lo or lo > sae_hi)
        sep[key] = {"overlap": bool(overlap), "dense_below_sae": bool(hi < sae_lo)}
        if overlap and n_breakeven is None:
            n_breakeven = n

    n_star = n_breakeven if isinstance(n_breakeven, int) else None
    K_star = (D_full / max(n_star, 1)) if n_star else None

    low_n_sep = any(sep.get(str(n), {}).get("dense_below_sae") for n in (1, 5))
    sae_strong = bool(np.isfinite(sae_val) and sae_val >= SAE_STRONG_THR)
    if not powered:
        verdict = "UNDERPOWERED"
    elif not sae_strong:
        verdict = "SAE_WEAK_DESCRIPTIVE"
    elif low_n_sep:
        verdict = "TRANSFER_CONFIRMED"
    elif (n_star is not None and n_star >= 5 and K_star is not None and K_star <= K_STAR_SMALL):
        verdict = "TRANSFER_VIA_BREAKEVEN"
    else:
        verdict = "NO_TRANSFER"
    return {"verdict": verdict, "n_breakeven": n_breakeven, "K_star": K_star,
            "ci_separation": sep, "low_n_sep": low_n_sep, "D_full": D_full}


transferB = {}
print("PIECE A/B — transfer verdicts on genuine deployment B (corpus_fold_B)\n")
for cid in cases:
    s = per_case[cid]["transferB"]
    rep = reproduce_transfer_verdict(s)
    transferB[cid] = rep
    match = "OK" if rep["verdict"] == s["verdict"] else "MISMATCH!"
    print("%-22s sae=%.3f  n*=%-4s K*=%-7s  -> %-22s  (reproduces stored: %s)"
          % (cid, s["sae_balacc"], rep["n_breakeven"], rep["K_star"], rep["verdict"], match))

## Fork verdict — `aggregate()`

The overall fork is `REAL_WHERE_TO_GATE_SAVING` iff **any** case is `TRANSFER_CONFIRMED` or
`TRANSFER_VIA_BREAKEVEN` on the genuine deployment B; otherwise the where-to-gate thesis is
honestly dropped. This is the **verbatim `aggregate()`** from `transfer.py` (the deployment-B branch).

In [ ]:
def aggregate(transferB):
    """Verbatim aggregate() from transfer.py (transferB branch)."""
    confirmed = [cid for cid, r in transferB.items()
                 if r.get("verdict") in ("TRANSFER_CONFIRMED", "TRANSFER_VIA_BREAKEVEN")]
    overall = "REAL_WHERE_TO_GATE_SAVING" if confirmed else "DROP_WHERE_TO_GATE"
    tallyB = {}
    for cid, r in transferB.items():
        tallyB.setdefault(r["verdict"], []).append(cid)
    return {"overall_transfer_fork_verdict": overall, "transfer_confirmed_on_B": confirmed,
            "transferB_tally": tallyB}

agg = aggregate(transferB)
print("OVERALL FORK :", agg["overall_transfer_fork_verdict"])
print("confirmed on B:", agg["transfer_confirmed_on_B"])
print("tally        :", {k: v for k, v in agg["transferB_tally"].items()})
if len(cases) == 5:
    print("\nmatches stored fork:", agg["overall_transfer_fork_verdict"] == md_block["overall_transfer_fork_verdict"])

## Piece C — Selection-independent behavioral-KL localization

The absorber was *selected* by firing precision (≈ balanced-accuracy), so we need a **different
axis** to claim it is behaviorally meaningful. Piece C ablates the absorber and measures the
next-token KL divergence on held-out rows: `targeting = mean(KL on absorbed-X) − mean(KL on
siblings)`. It is **localized** only if `targeting` is positive, its bootstrap CI excludes 0,
it exceeds the **90th-percentile** of a random-latent shuffle null, **and** at most 10% of
random latents match it. Rule below is **verbatim from `transfer.py:kl_targeting`**, recomputed
from the stored null draws — an honest split is expected: firing-balacc localizes a sense even
where behavioral-KL does not.

In [ ]:
def reproduce_kl_verdict(kl):
    """Verbatim localization rule from transfer.py:kl_targeting, recomputed from stored null_draws."""
    if kl is None or kl.get("status") != "ok":
        return {"verdict": "underpowered", "by_scale": {}}
    out = {}
    for sc, v in kl["by_scale"].items():
        targeting = v["targeting"]
        nulls = v["null_draws"]
        null_p90 = float(np.percentile(nulls, KL_NULL_PCTL)) if nulls else 0.0
        frac_null_ge = (float(np.mean([1.0 if x >= targeting else 0.0 for x in nulls]))
                        if nulls else 1.0)
        localized = bool(targeting > 0 and v["excl_0"] and targeting > null_p90
                         and frac_null_ge <= KL_FRAC_THR)
        out[sc] = {"verdict": ("KL_LOCALIZED" if localized else "KL_NULL_DESCRIPTIVE"),
                   "targeting": targeting, "null_p90": null_p90, "frac_null_ge": frac_null_ge}
    any_loc = any(x["verdict"] == "KL_LOCALIZED" for x in out.values())
    return {"verdict": "KL_LOCALIZED" if any_loc else "KL_NULL_DESCRIPTIVE", "by_scale": out}


kl_res = {}
print("PIECE C — behavioral-KL localization (scale=1.0 shown)\n")
for cid in cases:
    rep = reproduce_kl_verdict(per_case[cid]["kl_targeting"])
    kl_res[cid] = rep
    stored = per_case[cid]["kl_targeting"].get("verdict")
    match = "OK" if rep["verdict"] == stored else "MISMATCH!"
    sc1 = rep["by_scale"].get("1.0", {})
    print("%-22s targeting=%.4f  null_p90=%.4f  frac_ge=%.2f  -> %-20s (stored: %s)"
          % (cid, sc1.get("targeting", float("nan")), sc1.get("null_p90", float("nan")),
             sc1.get("frac_null_ge", float("nan")), rep["verdict"], match))

## Piece D — Amazon instrument-disagreement caveat

For the edit cases (Amazon, `large`) we diagnose whether the iter-9 `adv_joint`-vs-`adv_pres`
disagreement is **material**: if the judged forget gap between the SAE-handle (KG-ABL) route and
the dense-fair route exceeds `MATERIAL_THR` at matched behavioral forget, we soften
"demonstrated" to a preservation-advantage-only claim (`MATERIAL_REPORT_BOTH`); otherwise the
disagreement is isolated and immaterial (`ISOLATED_IMMATERIAL`). Rule **verbatim from
`transfer.py:amazon_caveat`**.

In [ ]:
def reproduce_caveat_verdict(cav):
    """Verbatim materiality rule from transfer.py:amazon_caveat."""
    if cav is None or cav.get("verdict") is None:
        return None
    gap = cav.get("forget_gap")
    immaterial = bool(gap is not None and abs(gap) < MATERIAL_THR)
    return {"verdict": "ISOLATED_IMMATERIAL" if immaterial else "MATERIAL_REPORT_BOTH",
            "forget_gap": gap, "fq_kg": cav.get("fq_kg"), "fq_fair": cav.get("fq_fair")}


print("PIECE D — edit instrument-disagreement caveat\n")
caveat_res = {}
for cid in cases:
    cav = per_case[cid].get("amazon_caveat")
    rep = reproduce_caveat_verdict(cav)
    if rep is None:
        continue
    caveat_res[cid] = rep
    match = "OK" if rep["verdict"] == cav["verdict"] else "MISMATCH!"
    print("%-22s forget_gap=%.3f (fq_kg=%s, fq_fair=%s)  -> %-22s (stored: %s)"
          % (cid, rep["forget_gap"], rep["fq_kg"], rep["fq_fair"], rep["verdict"], match))

# Honest negatives are computed once in the full run (build_honest_negatives needs eval-pool
# sizes not kept in per_case summaries), so we surface the stored list here.
print("\nHONEST NEGATIVES (stored):")
for s in md_block["honest_negatives"]:
    print(" -", s[:160])

## Results — summary table + transfer-curve visualization

A per-case summary table (transfer / KL / caveat verdicts), then the transfer curves: the dense
fair gate's balanced-accuracy vs label budget `n` (with 95% CIs) against the flat, label-free
fixed-id SAE handle (with its CI band). Where the dense curve sits **below** the SAE band at low
`n`, the SAE handle transfers with a label saving; where it overlaps early, dense "catches up".

In [ ]:
# ---------- summary table ----------
rows = []
for cid in cases:
    s = per_case[cid]["transferB"]
    tb = transferB[cid]
    kv = kl_res[cid]["verdict"]
    cav = caveat_res.get(cid, {}).get("verdict", "-")
    rows.append({
        "case": cid, "SAE_balacc": round(s["sae_balacc"], 3),
        "transfer_B": tb["verdict"], "n_breakeven": tb["n_breakeven"],
        "K*": tb["K_star"], "D_full": tb["D_full"], "KL": kv, "edit_caveat": cav,
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

# ---------- transfer-curve plots ----------
xs = list(range(len(N_GRID_T)))
xlabels = [str(n) for n in N_GRID_T]
ncol = min(len(cases), 3)
nrow = (len(cases) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.6 * nrow), squeeze=False)
for i, cid in enumerate(cases):
    ax = axes[i // ncol][i % ncol]
    s = per_case[cid]["transferB"]
    # dense fair gate curve (value + 95% CI) over n
    vals, los, his = [], [], []
    for n in N_GRID_T:
        dc = s["dense"].get(str(n))
        vals.append(dc["value"] if dc else np.nan)
        los.append((dc["value"] - dc["ci"][0]) if dc else 0.0)
        his.append((dc["ci"][1] - dc["value"]) if dc else 0.0)
    ax.errorbar(xs, vals, yerr=[los, his], marker="o", capsize=3, color="#1f77b4",
                label="dense fair gate (n labels)")
    # flat label-free SAE handle + CI band
    sv, (slo, shi) = s["sae_balacc"], s["sae_ci"]
    ax.axhline(sv, color="#d62728", ls="--", lw=1.6, label="fixed-id SAE handle (0 labels)")
    ax.axhspan(slo, shi, color="#d62728", alpha=0.12)
    ax.set_title("%s\n%s (n*=%s, K*=%s)" % (cid, transferB[cid]["verdict"],
                 transferB[cid]["n_breakeven"], transferB[cid]["K_star"]), fontsize=9)
    ax.set_xticks(xs); ax.set_xticklabels(xlabels)
    ax.set_xlabel("dense label budget n"); ax.set_ylabel("balanced accuracy")
    ax.set_ylim(0.45, 1.02); ax.grid(alpha=0.25)
    if i == 0:
        ax.legend(fontsize=7, loc="lower right")
# hide unused axes
for j in range(len(cases), nrow * ncol):
    axes[j // ncol][j % ncol].axis("off")
fig.suptitle("Cross-deployment B: label-free SAE handle vs dense gate fit on B's own n labels",
             fontsize=11, y=1.0)
fig.tight_layout()
plt.show()

# ---------- KL targeting vs random-latent null ----------
fig2, ax2 = plt.subplots(figsize=(1.6 * len(cases) + 2, 3.4))
tval = [kl_res[c]["by_scale"].get("1.0", {}).get("targeting", 0.0) for c in cases]
nval = [kl_res[c]["by_scale"].get("1.0", {}).get("null_p90", 0.0) for c in cases]
w = 0.38; idx = np.arange(len(cases))
ax2.bar(idx - w / 2, tval, w, label="absorber targeting", color="#2ca02c")
ax2.bar(idx + w / 2, nval, w, label="random-latent null (p90)", color="#999999")
ax2.set_yscale("symlog", linthresh=1e-3)
ax2.set_xticks(idx); ax2.set_xticklabels([c.split("_")[-1] for c in cases], rotation=20)
ax2.set_ylabel("KL targeting (scale=1.0, symlog)")
ax2.set_title("Piece C: behavioral-KL targeting vs null")
ax2.legend(fontsize=8); ax2.grid(alpha=0.25, axis="y")
fig2.tight_layout()
plt.show()

print("\nFORK:", agg["overall_transfer_fork_verdict"],
      "| KL tally:", {k: [c for c in cases if kl_res[c]["verdict"] == k]
                      for k in set(kl_res[c]["verdict"] for c in cases)})